<h3>Notebook 05 is the training notebook. We will keep it consistent with Notebook 03 and Notebook 04:</h3>
<ol>
<li> Input: 64 × 64 × 64 × 1 </li>
<li>3D ResUNet shared encoder</li>
<li>Segmentation output</li>
<li>Classification output</li>
<li>Batch size kept small to reduce Kaggle GPU/RAM pressure</li>
<li>Training/validation split</li>
<li>Early stopping</li>
<li>Best-model checkpoint</li>
<li>Learning-rate reduction</li>
<li>Training history</li>

<li>Separate monitoring of segmentation and classification</li>
</ol>
<h5>Keras supports callback-based training control, and multi-output models can be trained with separate losses and metrics for each output.</h5>

## 1. Import libraries


In [ ]:
# ============================================================
# NOTEBOOK 05
# 3D RESUNET TRAINING
# ============================================================

import os
import json
import random

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    Conv3D,
    BatchNormalization,
    Activation,
    Add,
    MaxPooling3D,
    UpSampling3D,
    Concatenate,
    GlobalAveragePooling3D,
    Dense,
    Dropout
)

In [ ]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print("Random seed:", SEED)

In [ ]:
# ============================================================
# CHECK GPU
# ============================================================

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU detected:")
    for gpu in gpus:
        print(gpu)
else:
    print("WARNING: No GPU detected.")

In [ ]:
# ============================================================
# GPU MEMORY CONFIGURATION
# ============================================================

gpus = tf.config.list_physical_devices("GPU")

if gpus:

    try:

        for gpu in gpus:
            tf.config.experimental.set_memory_growth(
                gpu,
                True
            )

        print("GPU memory growth enabled.")

    except RuntimeError as error:

        print(error)

In [ ]:
# ============================================================
# PATCH DATASET PATH
# ============================================================

PATCH_DATASET_ROOT = "/kaggle/input/YOUR-PATCH-DATASET-NAME"

X_PATH = os.path.join(
    PATCH_DATASET_ROOT,
    "X_patches.npy"
)

Y_SEG_PATH = os.path.join(
    PATCH_DATASET_ROOT,
    "Y_segmentation.npy"
)

Y_CLASS_PATH = os.path.join(
    PATCH_DATASET_ROOT,
    "Y_classification.npy"
)

print("X:", X_PATH)
print("Segmentation:", Y_SEG_PATH)
print("Classification:", Y_CLASS_PATH)

In [ ]:
# ============================================================
# VERIFY DATASET
# ============================================================

if not os.path.isfile(X_PATH):
    raise FileNotFoundError(
        f"X dataset not found:\n{X_PATH}"
    )

if not os.path.isfile(Y_SEG_PATH):
    raise FileNotFoundError(
        f"Segmentation labels not found:\n{Y_SEG_PATH}"
    )

if not os.path.isfile(Y_CLASS_PATH):
    raise FileNotFoundError(
        f"Classification labels not found:\n{Y_CLASS_PATH}"
    )

print("All training files found.")

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

X = np.load(X_PATH)

Y_seg = np.load(Y_SEG_PATH)

Y_class = np.load(Y_CLASS_PATH)

print("X shape:", X.shape)
print("Y_seg shape:", Y_seg.shape)
print("Y_class shape:", Y_class.shape)

In [ ]:
# ============================================================
# DATA TYPES
# ============================================================

X = X.astype(np.float32)

Y_seg = Y_seg.astype(np.float32)

Y_class = Y_class.astype(np.float32)

print("X dtype:", X.dtype)
print("Y_seg dtype:", Y_seg.dtype)
print("Y_class dtype:", Y_class.dtype)

In [ ]:
# ============================================================
# DATA VALIDATION
# ============================================================

assert X.ndim == 5

assert Y_seg.ndim == 5

assert Y_class.ndim == 1

assert X.shape[1:] == (
    64, 64, 64, 1
)

assert Y_seg.shape[1:] == (
    64, 64, 64, 1
)

assert X.shape[0] == Y_seg.shape[0]

assert X.shape[0] == Y_class.shape[0]

assert np.isfinite(X).all()

assert np.isfinite(Y_seg).all()

assert np.isfinite(Y_class).all()

print("Dataset validation successful.")

In [ ]:
# ============================================================
# CLASS DISTRIBUTION
# ============================================================

background_count = np.sum(
    Y_class == 0
)

fracture_count = np.sum(
    Y_class == 1
)

print("Background patches:", background_count)

print("Fracture patches:", fracture_count)

print(
    "Background percentage:",
    100 * background_count / len(Y_class)
)

print(
    "Fracture percentage:",
    100 * fracture_count / len(Y_class)
)

In [ ]:
# ============================================================
# DICE METRIC
# ============================================================

def dice_coefficient(y_true, y_pred):

    smooth = 1e-6

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred > 0.5,
        tf.float32
    )

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3, 4]
    )

    denominator = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3, 4]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3, 4]
        )
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        denominator + smooth
    )

    return tf.reduce_mean(dice)

In [ ]:
# ============================================================
# IOU METRIC
# ============================================================

def iou_coefficient(y_true, y_pred):

    smooth = 1e-6

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred > 0.5,
        tf.float32
    )

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3, 4]
    )

    union = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3, 4]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3, 4]
        )
        -
        intersection
    )

    iou = (
        intersection + smooth
    ) / (
        union + smooth
    )

    return tf.reduce_mean(iou)

In [ ]:
# ============================================================
# RESIDUAL BLOCK
# ============================================================

def residual_block(x, filters, name):

    shortcut = x

    x = Conv3D(
        filters,
        (3, 3, 3),
        padding="same",
        name=f"{name}_conv1"
    )(x)

    x = BatchNormalization(
        name=f"{name}_bn1"
    )(x)

    x = Activation(
        "relu",
        name=f"{name}_relu1"
    )(x)

    x = Conv3D(
        filters,
        (3, 3, 3),
        padding="same",
        name=f"{name}_conv2"
    )(x)

    x = BatchNormalization(
        name=f"{name}_bn2"
    )(x)

    if shortcut.shape[-1] != filters:

        shortcut = Conv3D(
            filters,
            (1, 1, 1),
            padding="same",
            name=f"{name}_shortcut"
        )(shortcut)

        shortcut = BatchNormalization(
            name=f"{name}_shortcut_bn"
        )(shortcut)

    x = Add(
        name=f"{name}_add"
    )([
        x,
        shortcut
    ])

    x = Activation(
        "relu",
        name=f"{name}_output"
    )(x)

    return x

In [ ]:
# ============================================================
# ENCODER BLOCK
# ============================================================

def encoder_block(x, filters, name):

    x = residual_block(
        x,
        filters,
        f"{name}_res"
    )

    skip = x

    x = MaxPooling3D(
        pool_size=(2, 2, 2),
        name=f"{name}_pool"
    )(x)

    return skip, x

In [ ]:
# ============================================================
# DECODER BLOCK
# ============================================================

def decoder_block(
    x,
    skip,
    filters,
    name
):

    x = UpSampling3D(
        size=(2, 2, 2),
        name=f"{name}_upsample"
    )(x)

    x = Concatenate(
        name=f"{name}_concat"
    )([
        x,
        skip
    ])

    x = residual_block(
        x,
        filters,
        f"{name}_res"
    )

    return x

In [ ]:
# ============================================================
# CLASSIFICATION BRANCH
# ============================================================

def classification_branch(
    bottleneck
):

    x = GlobalAveragePooling3D(
        name="classification_global_pool"
    )(bottleneck)

    x = Dense(
        128,
        activation="relu",
        name="classification_dense1"
    )(x)

    x = Dropout(
        0.3,
        name="classification_dropout"
    )(x)

    output = Dense(
        1,
        activation="sigmoid",
        name="classification_output"
    )(x)

    return output

In [ ]:
# ============================================================
# BUILD MULTI-TASK 3D RESUNET
# ============================================================

def build_3d_resunet(
    input_shape=(64, 64, 64, 1)
):

    inputs = Input(
        shape=input_shape,
        name="ct_input"
    )

    # --------------------------------------------------------
    # ENCODER
    # --------------------------------------------------------

    skip1, x = encoder_block(
        inputs,
        16,
        "encoder1"
    )

    skip2, x = encoder_block(
        x,
        32,
        "encoder2"
    )

    skip3, x = encoder_block(
        x,
        64,
        "encoder3"
    )

    # --------------------------------------------------------
    # BOTTLENECK
    # --------------------------------------------------------

    bottleneck = residual_block(
        x,
        128,
        "bottleneck"
    )

    # --------------------------------------------------------
    # CLASSIFICATION BRANCH
    # --------------------------------------------------------

    classification_output = (
        classification_branch(
            bottleneck
        )
    )

    # --------------------------------------------------------
    # DECODER
    # --------------------------------------------------------

    x = decoder_block(
        bottleneck,
        skip3,
        64,
        "decoder3"
    )

    x = decoder_block(
        x,
        skip2,
        32,
        "decoder2"
    )

    x = decoder_block(
        x,
        skip1,
        16,
        "decoder1"
    )

    # --------------------------------------------------------
    # SEGMENTATION OUTPUT
    # --------------------------------------------------------

    segmentation_output = Conv3D(
        1,
        (1, 1, 1),
        padding="same",
        activation="sigmoid",
        name="segmentation_output"
    )(x)

    # --------------------------------------------------------
    # FINAL MODEL
    # --------------------------------------------------------

    model = Model(
        inputs=inputs,
        outputs=[
            segmentation_output,
            classification_output
        ],
        name="3D_ResUNet_MultiTask"
    )

    return model

In [ ]:
# ============================================================
# CREATE MODEL
# ============================================================

model = build_3d_resunet(
    input_shape=(64, 64, 64, 1)
)

print("Model created.")

In [ ]:
# ============================================================
# COMPILE MODEL
# ============================================================

model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss={
        "segmentation_output":
            "binary_crossentropy",

        "classification_output":
            "binary_crossentropy"
    },

    loss_weights={
        "segmentation_output": 1.0,

        "classification_output": 1.0
    },

    metrics={
        "segmentation_output": [
            "accuracy",
            dice_coefficient,
            iou_coefficient
        ],

        "classification_output": [
            "accuracy",

            tf.keras.metrics.Precision(
                name="precision"
            ),

            tf.keras.metrics.Recall(
                name="recall"
            ),

            tf.keras.metrics.AUC(
                name="auc"
            )
        ]
    }
)

print("Model compiled successfully.")

In [ ]:
# ============================================================
# MODEL SUMMARY
# ============================================================

model.summary()

In [ ]:
# ============================================================
# TRAIN / VALIDATION SPLIT
# ============================================================

NUM_SAMPLES = len(X)

indices = np.arange(
    NUM_SAMPLES
)

rng = np.random.default_rng(
    SEED
)

rng.shuffle(
    indices
)

split_index = int(
    0.8 * NUM_SAMPLES
)

train_indices = indices[
    :split_index
]

val_indices = indices[
    split_index:
]

X_train = X[
    train_indices
]

X_val = X[
    val_indices
]

Y_seg_train = Y_seg[
    train_indices
]

Y_seg_val = Y_seg[
    val_indices
]

Y_class_train = Y_class[
    train_indices
]

Y_class_val = Y_class[
    val_indices
]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

In [ ]:
# ============================================================
# CHECK TRAINING CLASS DISTRIBUTION
# ============================================================

print("Training set")
print("--------------------------")

print(
    "Background:",
    np.sum(Y_class_train == 0)
)

print(
    "Fracture:",
    np.sum(Y_class_train == 1)
)

print()

print("Validation set")
print("--------------------------")

print(
    "Background:",
    np.sum(Y_class_val == 0)
)

print(
    "Fracture:",
    np.sum(Y_class_val == 1)
)

In [ ]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================

BATCH_SIZE = 1

EPOCHS = 20

LEARNING_RATE = 1e-4

print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)

In [ ]:
# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

OUTPUT_DIR = "/kaggle/working/training_results"

CHECKPOINT_DIR = os.path.join(
    OUTPUT_DIR,
    "checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

print("Output directory:")
print(OUTPUT_DIR)

In [ ]:
# ============================================================
# MODEL CHECKPOINT
# ============================================================

checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "best_3D_ResUNet.keras"
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(

    filepath=checkpoint_path,

    monitor="val_loss",

    save_best_only=True,

    save_weights_only=False,

    mode="min",

    verbose=1
)

In [ ]:
# ============================================================
# EARLY STOPPING
# ============================================================

early_stopping = tf.keras.callbacks.EarlyStopping(

    monitor="val_loss",

    patience=5,

    mode="min",

    restore_best_weights=True,

    verbose=1
)

In [ ]:
# ============================================================
# REDUCE LEARNING RATE
# ============================================================

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=2,

    min_lr=1e-6,

    verbose=1
)

In [ ]:
# ============================================================
# TRAINING LOG
# ============================================================

csv_log_path = os.path.join(
    OUTPUT_DIR,
    "training_history.csv"
)

csv_logger = tf.keras.callbacks.CSVLogger(
    csv_log_path
)

print("Training log:")
print(csv_log_path)

In [ ]:
# ============================================================
# TRAIN MODEL
# ============================================================

history = model.fit(

    X_train,

    {
        "segmentation_output":
            Y_seg_train,

        "classification_output":
            Y_class_train
    },

    validation_data=(

        X_val,

        {
            "segmentation_output":
                Y_seg_val,

            "classification_output":
                Y_class_val
        }
    ),

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    shuffle=True,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr,
        csv_logger
    ],

    verbose=1
)

In [ ]:
# ============================================================
# TRAINING HISTORY
# ============================================================

print("Available training metrics:")

for key in history.history.keys():
    print(key)

In [ ]:
# ============================================================
# TOTAL LOSS
# ============================================================

plt.figure(figsize=(8, 6))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("3D ResUNet Training and Validation Loss")

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# SEGMENTATION LOSS
# ============================================================

plt.figure(figsize=(8, 6))

plt.plot(
    history.history[
        "segmentation_output_loss"
    ],
    label="Training Segmentation Loss"
)

plt.plot(
    history.history[
        "val_segmentation_output_loss"
    ],
    label="Validation Segmentation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "Segmentation Training and Validation Loss"
)

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# CLASSIFICATION LOSS
# ============================================================

plt.figure(figsize=(8, 6))

plt.plot(
    history.history[
        "classification_output_loss"
    ],
    label="Training Classification Loss"
)

plt.plot(
    history.history[
        "val_classification_output_loss"
    ],
    label="Validation Classification Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "Classification Training and Validation Loss"
)

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# SEGMENTATION DICE
# ============================================================

plt.figure(figsize=(8, 6))

plt.plot(
    history.history[
        "segmentation_output_dice_coefficient"
    ],
    label="Training Dice"
)

plt.plot(
    history.history[
        "val_segmentation_output_dice_coefficient"
    ],
    label="Validation Dice"
)

plt.xlabel("Epoch")
plt.ylabel("Dice")

plt.title(
    "Segmentation Dice Coefficient"
)

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# SEGMENTATION IOU
# ============================================================

plt.figure(figsize=(8, 6))

plt.plot(
    history.history[
        "segmentation_output_iou_coefficient"
    ],
    label="Training IoU"
)

plt.plot(
    history.history[
        "val_segmentation_output_iou_coefficient"
    ],
    label="Validation IoU"
)

plt.xlabel("Epoch")
plt.ylabel("IoU")

plt.title(
    "Segmentation Intersection over Union"
)

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# CLASSIFICATION ACCURACY
# ============================================================

plt.figure(figsize=(8, 6))

plt.plot(
    history.history[
        "classification_output_accuracy"
    ],
    label="Training Accuracy"
)

plt.plot(
    history.history[
        "val_classification_output_accuracy"
    ],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title(
    "Classification Accuracy"
)

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# CLASSIFICATION AUC
# ============================================================

plt.figure(figsize=(8, 6))

plt.plot(
    history.history[
        "classification_output_auc"
    ],
    label="Training AUC"
)

plt.plot(
    history.history[
        "val_classification_output_auc"
    ],
    label="Validation AUC"
)

plt.xlabel("Epoch")
plt.ylabel("AUC")

plt.title(
    "Classification AUC"
)

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# LOAD BEST MODEL
# ============================================================

best_model = tf.keras.models.load_model(
    checkpoint_path,
    custom_objects={
        "dice_coefficient":
            dice_coefficient,

        "iou_coefficient":
            iou_coefficient
    }
)

print("Best model loaded successfully.")

In [ ]:
# ============================================================
# VALIDATION EVALUATION
# ============================================================

evaluation_results = best_model.evaluate(

    X_val,

    {
        "segmentation_output":
            Y_seg_val,

        "classification_output":
            Y_class_val
    },

    batch_size=BATCH_SIZE,

    verbose=1,

    return_dict=True
)

print("\nValidation results:")

for metric, value in evaluation_results.items():

    print(
        f"{metric}: {value:.6f}"
    )

In [ ]:
# ============================================================
# SAVE TRAINING HISTORY
# ============================================================

history_path = os.path.join(
    OUTPUT_DIR,
    "training_history.npy"
)

np.save(
    history_path,
    history.history,
    allow_pickle=True
)

print(
    "Training history saved:"
)

print(history_path)

In [ ]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================

final_model_path = os.path.join(
    OUTPUT_DIR,
    "final_3D_ResUNet_MultiTask.keras"
)

best_model.save(
    final_model_path
)

print(
    "Final model saved:"
)

print(final_model_path)

In [ ]:
# ============================================================
# SAVE EXPERIMENT CONFIGURATION
# ============================================================

experiment_config = {

    "model": "3D ResUNet Multi-Task",

    "input_shape": [
        64,
        64,
        64,
        1
    ],

    "patch_size": [
        64,
        64,
        64
    ],

    "batch_size": BATCH_SIZE,

    "epochs": EPOCHS,

    "learning_rate": LEARNING_RATE,

    "optimizer": "Adam",

    "segmentation_loss":
        "binary_crossentropy",

    "classification_loss":
        "binary_crossentropy",

    "segmentation_loss_weight":
        1.0,

    "classification_loss_weight":
        1.0,

    "train_samples":
        int(len(X_train)),

    "validation_samples":
        int(len(X_val)),

    "random_seed":
        SEED
}

config_path = os.path.join(
    OUTPUT_DIR,
    "experiment_config.json"
)

with open(
    config_path,
    "w"
) as f:

    json.dump(
        experiment_config,
        f,
        indent=4
    )

print(
    "Experiment configuration saved:"
)

print(config_path)

In [ ]:
# ============================================================
# FINAL TRAINING SUMMARY
# ============================================================

print("==============================================")
print("NOTEBOOK 05 - TRAINING COMPLETED")
print("==============================================")

print()

print("Training samples:")
print(len(X_train))

print()

print("Validation samples:")
print(len(X_val))

print()

print("Best model:")
print(checkpoint_path)

print()

print("Final validation metrics:")

for metric, value in evaluation_results.items():

    print(
        f"{metric}: {value:.6f}"
    )

print()

print("Training history:")
print(csv_log_path)

print("==============================================")